# 🛡️ Notebook 01: Mineração e Aquisição de Datasets em Escala Total
**Ambiente:** GPU 96 GB VRAM + 180 GB RAM (Google Colab High-Tier)
**Meta:** Coleta de 60.000 a 70.000 pares instrucionais estruturados nos 4 Pilares do Plano Diretor.

### Mapeamento dos Pilares:
1. **Pilar 1 (15.000 amostras):** Engenharia de Software e Baixo Nível (C, C++, Rust, Go, Python) via `Magicoder-Evol-Instruct`.
2. **Pilar 2 (22.000 amostras):** Patches SAST e Falhas Reais via `DiverseVul` e `PrimeVul` filtrados por CWEs críticas de memória.
3. **Pilar 3 (11.000 amostras):** Regras Estáticas de Defesa (Semgrep AST Rules, Assinaturas YARA e Sigma).


## 1. Verificação de Hardware e Montagem do Google Drive Persistente


In [ ]:
from google.colab import drive
import os, torch, psutil

drive.mount('/content/drive')

# Diagnóstico de Hardware
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
ram_gb = psutil.virtual_memory().total / (1024**3)

print('===========================================================')
print(f'🚀 GPU DETECTADA:  {gpu_name} ({vram_gb:.1f} GB VRAM)')
print(f'⚡ MEMÓRIA RAM:    {ram_gb:.1f} GB RAM do Sistema')
print(f'⚙️ NÚCLEOS CPU:    {os.cpu_count()} vCPUs')
print('===========================================================')

# Cria estrutura persistente no Drive
DRIVE_BASE = '/content/drive/MyDrive/cyber_llm_project'
RAW_DIR = os.path.join(DRIVE_BASE, 'raw_sources')
PROCESSED_DIR = os.path.join(DRIVE_BASE, 'processed_data')
TRAIN_DIR = os.path.join(DRIVE_BASE, 'training_ready')
CKPT_DIR = os.path.join(DRIVE_BASE, 'model_checkpoints')

for d in [RAW_DIR, PROCESSED_DIR, TRAIN_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✓ Diretórios no Google Drive sincronizados!')


In [ ]:
# Instalação de bibliotecas essenciais
!pip install -q datasets huggingface_hub pyarrow pandas tqdm fastparquet


## 2. Mineração do Pilar 2: Patches SAST (DiverseVul + PrimeVul - 22.000 amostras)
Extração via streaming focada nas CWEs críticas de corrupção de memória (CWE-119, CWE-120, CWE-125, CWE-416, CWE-787, CWE-476, CWE-190) e injeções (CWE-89, CWE-78, CWE-918).


In [ ]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

CRITICAL_CWES = {
    'CWE-119', 'CWE-120', 'CWE-125', 'CWE-787', 'CWE-416',
    'CWE-476', 'CWE-190', 'CWE-89', 'CWE-78', 'CWE-22', 'CWE-918', 'CWE-502'
}

print('[*] Iniciando streaming de DiverseVul...')
dataset_diverse = load_dataset('michelec/DiverseVul', split='train', streaming=True)

sast_records = []
target_sast = 22000
pbar = tqdm(total=target_sast, desc='Mineração SAST')

for item in dataset_diverse:
    cwes = item.get('cwe', [])
    if not isinstance(cwes, list):
        cwes = [cwes] if cwes else []
    
    if any(c in CRITICAL_CWES for c in cwes) and item.get('target') == 1:
        func = item.get('func', '').strip()
        if 40 < len(func) < 15000:
            sast_records.append({
                'id': f'diversevul-{len(sast_records):06d}',
                'category': 'sast_patch',
                'cwe_id': cwes[0],
                'language': 'c_cpp',
                'project': item.get('project', 'unknown'),
                'vulnerable_code': func,
                'fixed_code': ''
            })
            pbar.update(1)
            if len(sast_records) >= target_sast:
                break

pbar.close()
df_sast = pd.DataFrame(sast_records)
sast_path = os.path.join(RAW_DIR, 'diversevul_raw.parquet')
df_sast.to_parquet(sast_path, engine='pyarrow', compression='snappy')
print(f'✓ Salvas {len(df_sast):,} amostras de SAST no Google Drive: {sast_path}')


## 3. Mineração do Pilar 1: Engenharia de Software e Algoritmos de Baixo Nível (15.000 amostras)
Filtra algoritmos avançados, ponteiros, gerência de concorrência e sockets em C, C++, Rust, Go e Python.


In [ ]:
print('[*] Mineração do Magicoder-Evol-Instruct-110K...')
dataset_code = load_dataset('ise-uiuc/Magicoder-Evol-Instruct-110K', split='train', streaming=True)

code_records = []
target_code = 15000
pbar_code = tqdm(total=target_code, desc='Magicoder Low-Level')
keywords = ['c++', 'rust', 'golang', 'buffer', 'pointer', 'socket', 'concurrency', 'pthread', 'alloc', 'struct', 'memory', 'mutex']

for item in dataset_code:
    instr = item.get('instruction', '')
    resp = item.get('response', '')
    text = (instr + ' ' + resp).lower()
    
    if any(k in text for k in keywords):
        lang = 'c_cpp' if ('c++' in text or 'include <' in resp) else ('rust' if 'rust' in text else ('go' if 'golang' in text else 'python'))
        code_records.append({
            'id': f'gen-code-{len(code_records):06d}',
            'category': 'general_code',
            'cwe_id': 'None',
            'language': lang,
            'instruction': instr,
            'response': resp
        })
        pbar_code.update(1)
        if len(code_records) >= target_code:
            break

pbar_code.close()
df_code = pd.DataFrame(code_records)
code_path = os.path.join(RAW_DIR, 'magicoder_code.parquet')
df_code.to_parquet(code_path, engine='pyarrow', compression='snappy')
print(f'✓ Salvas {len(df_code):,} amostras de código geral: {code_path}')


## 4. Mineração do Pilar 3: Regras Estáticas Semgrep e Assinaturas YARA (11.000 amostras)


In [ ]:
!mkdir -p /content/cache/rule_repos
%cd /content/cache/rule_repos
!git clone --depth 1 https://github.com/semgrep/semgrep-rules.git
!git clone --depth 1 https://github.com/Yara-Rules/rules.git yara-rules

import glob, re

rule_records = []

# Regras Semgrep
semgrep_files = glob.glob('/content/cache/rule_repos/semgrep-rules/**/*.y*ml', recursive=True)
for yf in semgrep_files[:7000]:
    try:
        with open(yf, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        if 'pattern' in content and 'rules:' in content:
            rule_records.append({
                'id': f'semgrep-{len(rule_records):06d}',
                'category': 'detection_rule',
                'cwe_id': 'CWE-StaticRule',
                'language': 'semgrep',
                'instruction': 'Escreva uma regra Semgrep para analisar AST e detectar vulnerabilidade de código.',
                'response': f'```yaml\n{content.strip()}\n```'
            })
    except Exception:
        continue

# Assinaturas YARA
yara_files = glob.glob('/content/cache/rule_repos/yara-rules/**/*.yar*', recursive=True)
for yf in yara_files[:5000]:
    try:
        with open(yf, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        if 'rule ' in content and 'strings:' in content:
            rule_records.append({
                'id': f'yara-{len(rule_records):06d}',
                'category': 'detection_rule',
                'cwe_id': 'Malware-Signature',
                'language': 'yara',
                'instruction': 'Elabore uma regra YARA estrita para detecção de binários maliciosos.',
                'response': f'```yara\n{content.strip()}\n```'
            })
    except Exception:
        continue

df_rules = pd.DataFrame(rule_records)
rules_path = os.path.join(RAW_DIR, 'detection_rules.parquet')
df_rules.to_parquet(rules_path, engine='pyarrow', compression='snappy')
print(f'✓ Salvas {len(df_rules):,} regras defensivas: {rules_path}')


## 5. Resumo da Coleta Consolidada no Google Drive


In [ ]:
import glob
parquet_files = glob.glob(f'{RAW_DIR}/*.parquet')
total = 0
print('=== VOLUMETRIA TOTAL MINERADA (GOOGLE DRIVE) ===')
for pf in parquet_files:
    df = pd.read_parquet(pf)
    total += len(df)
    print(f'• {os.path.basename(pf)}: {len(df):,} registros')
print(f'\nTOTAL GERAL CONSOLIDADO: {total:,} amostras!')
print('-> Prossiga para o Notebook 02 para Higienização, Validação e CoT!')
